# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditi-avni/ML-FlyRank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
!git clone https://github.com/aditi-avni/ML-FlyRank.git

Cloning into 'ML-FlyRank'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 145 (delta 55), reused 93 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1.85 MiB | 16.22 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [5]:
!pip install -q pandas numpy scikit-learn matplotlib duckdb

In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 30000
Columns: 44


In [18]:
import numpy as np

# Numeric features available before prediction
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Categorical features available before prediction
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

# Build raw feature vector
X = df[numeric_features + categorical_features].copy()

# Log-transformed count features
X["log_impressions_90d"] = np.log1p(X["impressions_90d"])
X["log_clicks_90d"] = np.log1p(X["clicks_90d"])
X["log_sessions_90d"] = np.log1p(X["sessions_90d"])
X["log_ai_sessions_90d"] = np.log1p(X["ai_sessions_90d"])

# Missingness flags
missing_flag_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "scroll_rate"
]

for col in missing_flag_cols:
    X[f"has_{col}"] = X[col].notna().astype(int)

# Fill numeric missing values with median
for col in missing_flag_cols:
    X[col] = X[col].fillna(X[col].median())

# Fill categorical missing values with explicit category
categorical_missing_cols = [
    "competition_level",
    "main_intent",
    "word_count_tier",
    "char_count_tier"
]

for col in categorical_missing_cols:
    X[col] = X[col].fillna("missing")

print("Feature matrix shape:", X.shape)
print("Remaining missing values:", X.isna().sum().sum())

Feature matrix shape: (30000, 37)
Remaining missing values: 0


## 2. Feature notes (meaning, missing, categorical, available-when?)

### Feature notes

| Feature group | Meaning | Missing-value handling | Available before prediction? |
|---|---|---|---|
| Search features | Search demand and competition around the content | Median fill + missingness flag where applicable | Yes |
| Content features | Word count, character count and content type/intent | Median fill for numeric values; `"missing"` category for categorical values | Yes |
| Historical performance | Impressions, clicks, sessions and AI sessions over the historical period | No missing values in selected fields | Yes |
| Activity features | Days with impressions/sessions and content age | No missing values in selected fields | Yes |
| Rate features | CTR, average position, engagement, scroll and AI traffic rates | Scroll rate gets median fill + missingness flag | Yes |
| Log features | Log-transformed versions of large count variables | Derived from historical count features | Yes |
| Missingness flags | Indicates whether selected numeric information was originally missing | 1 = present, 0 = missing | Yes |

The feature vector excludes identifiers and fields that directly describe the outcome.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [19]:
# Target label
y = (df["trend_direction"] == "down").astype(int)

print("Label counts:")
print(y.value_counts())

print("\nLabel proportions:")
print(y.value_counts(normalize=True))

Label counts:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Label proportions:
trend_direction
1    0.542067
0    0.457933
Name: proportion, dtype: float64


In [20]:
leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

print("Leakage / exclusion candidates:")
for col in leakage_candidates:
    print(col, "->", col in X.columns)

Leakage / exclusion candidates:
trend_direction -> False
trend_pct -> False
content_id -> False
client_id -> False


In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Deliberately create a leaky feature from the label
X_leaky = X.copy()
X_leaky["LEAKY_LABEL"] = y

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Use only numeric columns for this simple demonstration
X_train_num = X_train.select_dtypes(include=np.number)
X_test_num = X_test.select_dtypes(include=np.number)

leak_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leak_model.fit(X_train_num, y_train)

leak_predictions = leak_model.predict(X_test_num)

print("Accuracy with deliberate leaky feature:",
      accuracy_score(y_test, leak_predictions))

Accuracy with deliberate leaky feature: 1.0


In [22]:
X = X.drop(columns=["LEAKY_LABEL"], errors="ignore")

print("LEAKY_LABEL present after removal:",
      "LEAKY_LABEL" in X.columns)

LEAKY_LABEL present after removal: False


## 4. What I excluded and why

### What I excluded and why

- `content_id` — identifier only; does not represent a content characteristic.
- `client_id` — identifier used for grouping/splitting, not a predictive feature.
- `trend_direction` — directly defines the decline outcome, so using it would leak the label.
- `trend_pct` — directly contributes to the construction of `trend_direction`, so it is label-derived and would leak outcome information.

I also excluded product/decision-derived fields from the feature vector because they may encode decisions made after observing performance rather than information available at the prediction moment.

The selected features therefore focus on content characteristics, search context, historical performance and engagement signals that can be observed before making the prediction.


In [23]:
print("Final feature matrix:", X.shape)

print("\nRemaining missing values:")
print(X.isna().sum().sum())

print("\nExcluded columns present in X:")
for col in leakage_candidates:
    print(col, "->", col in X.columns)

Final feature matrix: (30000, 37)

Remaining missing values:
0

Excluded columns present in X:
trend_direction -> False
trend_pct -> False
content_id -> False
client_id -> False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.